# Adversarial validation

The final CNN's architecture (`src/best_model.py`) retrained to tell test photos from train photos (`src/adversarial.py`):
5 folds that keep pHash near-duplicates together, 10 epochs per fold, one out-of-fold AUC over train + test.

Run on a Colab T4. Before you start, upload `av_code.zip` (the code and `results/adversarial/folds.csv`) and the Kaggle
credentials: `.env` with `KAGGLE_USERNAME` and `KAGGLE_KEY`, or `kaggle.json`.

Results go to Drive, `dl_lab2/results/adversarial/`: `oof_fold*.csv`, `oof.csv`, `summary.json`, `likeness_by_label.csv`,
`image_stats.csv`, `mlflow.db` and the figures.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path("/content/CNN_for_multilabel_classification")
DRIVE_RESULTS = Path("/content/drive/MyDrive/dl_lab2/results")
LOG = Path("/content/adversarial.log")


def uploaded(name):
    """Where 'Upload to Colab' put a file: /content or one folder below it."""
    hits = [*Path("/content").glob(name), *Path("/content").glob(f"*/{name}")]
    hits = [p for p in hits if p.parts[2] not in ("drive", REPO.name)]
    return min(hits, key=lambda p: len(p.parts), default=None)


if not REPO.exists():
    shutil.unpack_archive(uploaded("av_code.zip"), REPO)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
packages = ["lightning", "mlflow", "albumentations>=2.0.8", "imagehash", "torchmetrics", "kaggle"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

from google.colab import drive

drive.mount("/content/drive")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
os.environ["RESULTS_DIR"] = str(DRIVE_RESULTS)  # read by src.config at import
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

from src import colab
from src.adversarial import OUT_DIR
from src.config import DATA_DIR
from src.data import build_cache

OUT_DIR.mkdir(parents=True, exist_ok=True)
if not (OUT_DIR / "folds.csv").exists():
    shutil.copy(REPO / "results" / "adversarial" / "folds.csv", OUT_DIR / "folds.csv")
print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout)

In [ ]:
if creds := uploaded(".env"):
    colab.load_env(creds)
elif creds := uploaded("kaggle.json"):
    (Path.home() / ".kaggle").mkdir(exist_ok=True)
    shutil.copy(creds, Path.home() / ".kaggle" / "kaggle.json")
    (Path.home() / ".kaggle" / "kaggle.json").chmod(0o600)
colab.ensure_dataset(colab.DRIVE_DIR)
for split in ("train", "test"):
    build_cache(split, 224)
print({s: len(os.listdir(DATA_DIR / s)) for s in ("train", "test")})

## Train

In [ ]:
def running():
    return subprocess.run(["pgrep", "-f", "src.adversarial"], capture_output=True).returncode == 0


if not running():
    job = subprocess.Popen([sys.executable, "-m", "src.adversarial"], cwd=REPO, stdout=LOG.open("ab"),
                           stderr=subprocess.STDOUT, start_new_session=True)

## Progress

In [ ]:
def progress(lines=4):
    folds = sorted(p.stem.removeprefix("oof_") for p in OUT_DIR.glob("oof_fold*.csv"))
    print("running" if running() else "stopped", "| folds done:", folds)
    rows = [r.strip() for r in LOG.read_text(errors="replace").replace("\r", "\n").splitlines() if r.strip()]
    print(*[r for r in rows if r.startswith("[done]")], *rows[-lines:], sep="\n")


progress()

## Results

In [ ]:
import pandas as pd

if not running() and (OUT_DIR / "likeness_by_label.csv").exists():
    for name in ("av_hist.png", "av_extremes.png"):
        shutil.copy(REPO / "docs" / "figures" / name, OUT_DIR / name)
    shutil.copy(LOG, OUT_DIR / "run.log")
    print((OUT_DIR / "summary.json").read_text())
    display(pd.read_csv(OUT_DIR / "likeness_by_label.csv", index_col=0))